# World map prototype for VizAvalanche

## Install packages and load libraries

In [ ]:
# !pip install pandas plotly
# !pip install nbformat

# libraries
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

You should consider upgrading via the '/Users/lichuinchong/.pyenv/versions/3.10.1/bin/python3.10 -m pip install --upgrade pip' command.


## CSV path

In [7]:
CSV_PATH = Path("data/collected_data.csv")

df = pd.read_csv(CSV_PATH)
print(f"Loaded: {CSV_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

Loaded: data/collected_data.csv
Rows: 13,049,920
Columns: 11


,Unnamed: 0,is_cluster,molecule_type,ncbi_family,ncbi_genus,ncbi_species,sequence_gc_content,sequence_length,sequence_title,sequence_completeness,submitter_country
0,0,True,dsDNA,NaN,NaN,Caudoviricetes sp.,NaN,14449,"MAG TPA_asm: Podoviridae sp. isolate ctU6X1, p...",partial,USA
1,1,True,dsDNA,NaN,NaN,Caudoviricetes sp.,NaN,12855,"MAG TPA_asm: Siphoviridae sp. isolate ctoxf14,...",partial,USA
2,2,True,dsDNA,NaN,NaN,Caudoviricetes sp.,NaN,39803,"MAG TPA_asm: Myoviridae sp. isolate ctaaE5, pa...",partial,USA
3,3,True,dsDNA,NaN,NaN,Caudoviricetes sp.,NaN,64002,"MAG TPA_asm: Podoviridae sp. isolate ctxdM6, p...",partial,USA
4,4,True,dsDNA,NaN,NaN,Caudoviricetes sp.,NaN,116470,"MAG TPA_asm: Siphoviridae sp. isolate ct3B41, ...",partial,USA


In [8]:
# inspect the column
df["submitter_country"].isna().sum() # 192
df["submitter_country"].value_counts()

submitter_country
USA               5497357
United Kingdom    2807681
Germany            863844
Denmark            510447
China              477251
                   ...   
West Bank               2
Fiji                    2
Albania                 2
USSR                    1
Tajikistan              1
Name: count, Length: 164, dtype: int64

## country counts

In [11]:
country_counts = (
    df["submitter_country"]
    .value_counts()
    .reset_index()
)

country_counts.columns = ["country", "count"]
country_counts

# log count
import numpy as np

country_counts["log_count"] = np.log10(country_counts["count"] + 1)
country_counts.head()

,country,count,log_count
0,USA,5497357,6.740154
1,United Kingdom,2807681,6.448348
2,Germany,863844,5.936436
3,Denmark,510447,5.707952
4,China,477251,5.678748


## World map: country-level choropleth

In [14]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

max_count = country_counts["count"].max()

tick_counts = [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000,]
if max_count >= 5_000_000:
    tick_counts.append(5_000_000)
if max_count >= 10_000_000:
    tick_counts.append(10_000_000)
tick_counts = [x for x in tick_counts if x <= max_count or x == 1]

tick_vals = [np.log10(x + 1) for x in tick_counts]

def format_count_label(x):
    if x >= 1_000_000:
        return f"{x // 1_000_000}M"
    elif x >= 1_000:
        return f"{x // 1_000}k"
    else:
        return str(x)
tick_text = [format_count_label(x) for x in tick_counts]

fig_map = go.Figure(
    data=go.Choropleth(
        locations=country_counts["country"],
        z=country_counts["log_count"],
        locationmode="country names",
        text=country_counts["country"],
        customdata=country_counts[["count"]],
        colorbar=dict(
            title="Number of records",
            tickvals=tick_vals,
            ticktext=tick_text,
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Records: %{customdata[0]:,}<br>"
            "<extra></extra>"
        ),
        marker_line_color="white",
        marker_line_width=0.5,
    )
)

fig_map.update_layout(
    title="World map of submitter countries (log-scaled by record count)",
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth",
    ),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig_map.show()

/var/folders/ht/fy5c__851734w0h7g4bd7rnw0000gn/T/ipykernel_19454/3332942551.py:26: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = go.Figure(
